In [1]:
pip install mysql-connector-python sastrawi sentence-transformers faiss-cpu bertopic

   ---------------------------------------- 0.0/726.2 kB ? eta -:--:--
   ---------------------------------------- 726.2/726.2 kB 5.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   --- ------------------------------------ 1.3/16.3 MB 6.7 MB/s eta 0:00:03
   -------- ------------------------------- 3.4/16.3 MB 8.7 MB/s eta 0:00:02
   -------------- ------------------------- 6.0/16.3 MB 10.5 MB/s eta 0:00:01
   --------------------- ------------------ 8.9/16.3 MB 11.3 MB/s eta 0:00:01
   ------------------------------ --------- 12.6/16.3 MB 12.5 MB/s eta 0:00:01
   -------------------------------------- - 15.7/16.3 MB 13.2 MB/s eta 0:00:01
   ---------------------------------------- 16.3/16.3 MB 13.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import mysql.connector
import re
import numpy as np
import faiss
import torch
from sklearn.decomposition import PCA
from bertopic import BERTopic
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sentence_transformers import SentenceTransformer, util

In [3]:
def fetch_projects():
    conn = mysql.connector.connect(
        host='localhost',
        user='root',
        password='data@123',
        database='monitor-pa-ml'
    )
    cursor = conn.cursor(dictionary=True)
    query = """
        SELECT data_pa_id, judul_pa, judul_pa_en, desc_pa, platform_aplikasi, teknologi_yg_digunakan
        FROM data_pa
    """
    cursor.execute(query)
    projects = cursor.fetchall()
    cursor.close()
    conn.close()
    return projects


In [53]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import re

# Create stemmer and stopword remover
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stopword_factory = StopWordRemoverFactory()
stopwords = set(stopword_factory.get_stop_words())

# Add domain-specific stopwords
custom_stopwords = {'aplikasi', 'berbasis', 'android', 'untuk', 'dan'}

def preprocess(text):
    if not text:
        return ""
    # Lowercase and clean text
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize and remove stopwords
    words = text.split()
    words = [word for word in words if word not in stopwords and word not in custom_stopwords]
    text = ' '.join(words)
    
    # Stemming
    text = stemmer.stem(text)
    
    return text


In [54]:
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
indobert_model = SentenceTransformer("indobenchmark/indobert-base-p1")

No sentence-transformers model found with name indobenchmark/indobert-base-p1. Creating a new one with mean pooling.


In [73]:
from sklearn.preprocessing import normalize

def generate_combined_embeddings(projects):
    texts = []
    for project in projects:
        text = f"{project['judul_pa'] or ''} {project['judul_pa_en'] or ''} {project['desc_pa'] or ''}"
        text = preprocess(text)
        texts.append(text)
    
    sbert_embeddings = sbert_model.encode(texts, convert_to_numpy=True)
    indobert_embeddings = indobert_model.encode(texts, convert_to_numpy=True)

    combined_embeddings = np.hstack([sbert_embeddings, indobert_embeddings])
    combined_embeddings = normalize(combined_embeddings, norm='l2').astype('float32')  # Normalize once here

    return texts, combined_embeddings


In [64]:
def run_topic_modeling(texts, embeddings):
    topic_model = BERTopic(language="indonesian", 
                       min_topic_size=5, 
                       nr_topics="auto", 
                       calculate_probabilities=True, 
                       verbose=True)

    topics, _ = topic_model.fit_transform(texts, embeddings)
    return topic_model, topics


In [104]:
topic_model.save("saved_models/topic_model")


2025-05-30 00:59:20,805 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [74]:
def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # inner product = cosine similarity if normalized
    index.add(embeddings)
    return index


In [76]:
def find_similar_projects(input_text, combined_embeddings, projects, faiss_index, topics, topic_model, indobert_model, sbert_model, top_k=5):
    input_text = preprocess(input_text)  # Preprocess input text
    indo_embedding = indobert_model.encode([input_text], convert_to_numpy=True)
    sbert_embedding = sbert_model.encode([input_text], convert_to_numpy=True)

    combined_embedding = np.hstack([sbert_embedding, indo_embedding])
    combined_embedding = normalize(combined_embedding, norm='l2').astype('float32')

    distances, indices = faiss_index.search(combined_embedding, top_k)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        topic_id = topics[idx]
        topic_keywords = topic_model.get_topic(topic_id)
        keywords = ', '.join([kw for kw, _ in topic_keywords[:5]]) if topic_keywords else "N/A"

        results.append({
            'data_pa_id': projects[idx]['data_pa_id'],
            'judul_pa': projects[idx]['judul_pa'],
            'judul_pa_en': projects[idx]['judul_pa_en'],
            'desc_pa': projects[idx]['desc_pa'],
            'platform_aplikasi': projects[idx]['platform_aplikasi'],
            'teknologi_yg_digunakan': projects[idx]['teknologi_yg_digunakan'],
            'topic_id': topic_id,
            'topic_keywords': keywords,
            'similarity_score': float(dist)  # cosine similarity in [0, 1]
        })

    return results


In [68]:
projects = fetch_projects()

In [99]:
# Save the fetched project data
import pickle
with open("projects.pkl", "wb") as f:
    pickle.dump(projects, f)

In [77]:

texts, combined_embeddings = generate_combined_embeddings(projects)
topic_model, topics = run_topic_modeling(texts, combined_embeddings)
faiss_index = build_faiss_index(combined_embeddings)


2025-05-29 23:28:51,951 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-05-29 23:28:52,436 - BERTopic - Dimensionality - Completed ✓
2025-05-29 23:28:52,443 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-29 23:28:52,508 - BERTopic - Cluster - Completed ✓
2025-05-29 23:28:52,508 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-05-29 23:28:52,568 - BERTopic - Representation - Completed ✓
2025-05-29 23:28:52,568 - BERTopic - Topic reduction - Reducing number of topics
2025-05-29 23:28:52,576 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-05-29 23:28:52,617 - BERTopic - Representation - Completed ✓
2025-05-29 23:28:52,624 - BERTopic - Topic reduction - Reduced number of topics from 30 to 19


In [87]:
new_title = "Aplikasi Mengenal Huruf Hijaiyah"
new_title_en = "Arabic Alphabets Learning Application"
new_desc = """pada usia 4-6 tahun anak sudah mulai dikenalkan dan diajarkan mengenai cara baca Al-Quran, tata cara shalat, dan juga doa-doa dalam beraktivitas sehari-hari. 
Secara garis besar dalam pembelajaran terdapat tiga gaya belajar yaitu visual, auditorial, dan kinestetik. Yang mana, gaya belajar ini adalah metode terbaik untuk mengumpulkan dan menggunakan pengetahuan secara spesifik. Agar, proses belajar menyenangkan dan menumbuhkan daya ingat anak maka dapat menggabungkan beberapa gaya belajar yang disebut dengan metode Vizualitation Auditory Kinestetic (VAK).
Aplikasi ini merupakan aplikasi yang dirancang sebagai media pembelajaran untuk anak usia dini berupa pengenalan  huruf hijaiyah dengan gambar dan suara, pengenalan huruf hijaiyah dalam bentuk puzzle, dan juga pengenala huruf hijaiyah dalam bentuk AR (Augmented Reality)."""

new_input_text = f"{new_title} {new_title_en} {new_desc}"


In [88]:
similar_projects = find_similar_projects(
    new_input_text,
    combined_embeddings,
    projects,
    faiss_index,
    topics,
    topic_model,
    indobert_model,   # pass this model
    sbert_model,      # pass this model
    top_k=10
)


In [89]:
for proj in similar_projects:
    print(f"{proj['judul_pa']} | Topic {proj['topic_id']} | Score: {proj['similarity_score']:.3f}")
    print(f"  ➤ Keywords: {proj['topic_keywords']}")
    print()

Aplikasi Mengenal Huruf Hijaiyah | Topic 1 | Score: 1.000
  ➤ Keywords: main, game, tradisional, anak, budaya

FunRecite: Aplikasi Belajar Mengaji Al-Qur'an Untuk Anak Berbasis Augmented Reality | Topic 17 | Score: 0.854
  ➤ Keywords: isyarat, bahasa, tunarungu, baca, huruf

INDOPHONIC: APLIKASI BELAJAR MEMBACA BAHASA INDONESIA MENGGUNAKAN METODE FONIK BERBASIS ANDROID | Topic 17 | Score: 0.832
  ➤ Keywords: isyarat, bahasa, tunarungu, baca, huruf

Dislect: Aplikasi pembelajaran penyandang disleksia dengan metode hand drawing | Topic -1 | Score: 0.812
  ➤ Keywords: guna, laku, data, informasi, reality

Aplikasi Pelafalan Huruf Hijaiyah Untuk Anak-anak Berbasis Augmented Reality | Topic -1 | Score: 0.809
  ➤ Keywords: guna, laku, data, informasi, reality

Aplikasi Permainan Montessori untuk  Stimulus Perkembangan Anak | Topic 1 | Score: 0.809
  ➤ Keywords: main, game, tradisional, anak, budaya

Easy English : Game Belajar Bahasa Inggris Berbasis Android Menggunakan Metode Phonic | Topic

In [90]:
topic_model.visualize_barchart(top_n_topics=10)


In [91]:
topic_model.visualize_term_rank()


In [92]:
fig = topic_model.visualize_topics()
fig.write_html("topics.html")


saving parts


In [105]:
from bertopic import BERTopic

topic_model = BERTopic.load("saved_models/topic_model")


In [103]:
import pickle
import numpy as np
import os

# Create a directory to store saved files
os.makedirs("saved_models", exist_ok=True)

# Save the topic model using BERTopic’s save method (recommended)
topic_model.save("saved_models/topic_model")

# Save other components
with open("saved_models/projects.pkl", "wb") as f:
    pickle.dump(projects, f)

np.save("saved_models/combined_embeddings.npy", combined_embeddings)
np.save("saved_models/topics.npy", topics)


2025-05-30 00:44:54,889 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [93]:
topic_model.save("bertopic_model")


2025-05-30 00:15:11,543 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [94]:
faiss.write_index(faiss_index, "faiss_index.index")


In [95]:
np.save("combined_embeddings.npy", combined_embeddings)


In [96]:
import json

with open("topics.json", "w") as f:
    json.dump(topics, f)


In [97]:
from bertopic import BERTopic
import faiss
import numpy as np
import json

topic_model = BERTopic.load("bertopic_model")
faiss_index = faiss.read_index("faiss_index.index")
combined_embeddings = np.load("combined_embeddings.npy")

with open("topics.json", "r") as f:
    topics = json.load(f)


In [100]:
import numpy as np
import pickle

# These must be already created in the notebook
with open("projects.pkl", "wb") as f:
    pickle.dump(projects, f)

np.save("combined_embeddings.npy", combined_embeddings)
np.save("topics.npy", topics)

with open("topic_model.pkl", "wb") as f:
    pickle.dump(topic_model, f)


In [101]:
topic_model.save("topic_model")  # Saves as a folder


2025-05-30 00:39:12,521 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [102]:
from bertopic import BERTopic
topic_model = BERTopic.load("topic_model")
